# Task 2 · Retrieval Trigger (When to Use RAG)
**Owner:** Manimegalai Kumar Periyasamy · DSA 4 (Action Learning)

**Research question:** When does an LLM need external retrieval (RAG) before answering, which trigger matches always-RAG accuracy with the fewest retrievals, and do thresholds tuned on single-hop questions (**TriviaQA**) transfer unchanged to multi-hop questions (**HotpotQA**)?

**Kaggle settings:** **GPU T4** (needed for V4 DistilBERT fine-tuning; V1–V3 run offline on CPU), **Internet on** (to pull `distilbert-base-uncased` and the team repo). **Add Input** → `dsa4-data` and `dsa4-cache`.

## 1. Introduction

In a production LLM question-answering pipeline, *always retrieving* external evidence wastes retrieval and prompt-token budgets on facts the model already knows (and can inject distracting passages), while *never retrieving* causes hallucinations on unfamiliar or multi-hop questions.

Following **Moskvoretskii et al. (ACL 2025)** and **Jeong et al. (NAACL 2024, Adaptive-RAG)**, this notebook compares four decision policies under a strict fair-comparison protocol (identical generator `Qwen2.5-7B-Instruct`, identical prompt, identical `bge-base-en-v1.5` + FAISS top-5 index, and **no semantic cache** in Task 2):

- **V1 Fixed baselines:** Never retrieve (`V1a`) and Always retrieve (`V1b`).
- **V2 Token log-probability (post-generation):** Retrieve if the length-normalised log-probability of the direct answer falls below a threshold.
- **V3 Predictive entropy (post-generation):** Retrieve if the mean token entropy of the direct answer exceeds a threshold.
- **V4 Complexity classifier (pre-generation):** `DistilBERT-base` (66M) fine-tuned on question text to predict silver labels (*did retrieval fix a wrong direct answer?*).

**Hypothesis:** Output-uncertainty triggers (V2, V3) will approach always-retrieve accuracy on TriviaQA with far fewer retrievals, but under-retrieve on HotpotQA due to overconfidence on multi-hop questions; the pre-generation DistilBERT classifier (V4) avoids paying two LLM passes on triggered queries and is tested for stronger transfer.

## 2. Setup and data

In [ ]:
# Setup: clone/update the team repository and locate Kaggle inputs
import glob, os, subprocess, sys

REPO_URL = "https://github.com/Naveena126/dsa4-reuse-retrieve-generate.git"
REPO_BRANCH = "develop"  # switch to e.g. "task2-retrieval-trigger" before PR merge, or "main" at defense

if os.path.exists("/kaggle/working"):
    repo_dir = "/kaggle/working/repo"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, REPO_URL, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "fetch", "-q", "origin"], check=True)
        subprocess.run(["git", "-C", repo_dir, "checkout", "-q", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", repo_dir, "pull", "-q", "origin", REPO_BRANCH], check=True)
    os.chdir(repo_dir)

sys.path.insert(0, os.path.abspath("src"))

def find_input(name):
    # Kaggle mounts datasets at /kaggle/input/<name> or /kaggle/input/datasets/<owner>/<name>
    hits = [h for h in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if os.path.isdir(h)]
    if not hits:
        raise FileNotFoundError(f"Add the dataset '{name}' via Kaggle Add Input")
    return hits[0]

import torch
print("Working dir:", os.getcwd())
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      f"({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dsa4 import cost, data, evaluate, retrieval

MET, FIG = Path("results/metrics"), Path("results/figures")
MET.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

# Locate questions.jsonl (dsa4-data) and cache.jsonl (dsa4-cache)
DATA_DIR = Path(glob.glob(find_input("dsa4-data") + "/**/questions.jsonl", recursive=True)[0]).parent
CACHE_PATH = glob.glob(find_input("dsa4-cache") + "/**/cache.jsonl", recursive=True)[0]

q_df = data.load_questions(DATA_DIR)
cache_df = pd.read_json(CACHE_PATH, lines=True)
df = retrieval.prepare_retrieval_frame(q_df, cache_df)

print(f"Loaded {len(df)} source questions from {CACHE_PATH}")
display(df.groupby("dataset")[["split_s0"]].value_counts().unstack(fill_value=0))

## 3. Data exploration and silver labels

For every source question we have both the direct answer and the top-5 RAG answer from the shared cache, scored against all accepted aliases with `dsa4.evaluate` (normalised Exact Match & token F1).

Silver labels for Task 2:
- **`retrieval_helps` (`RAG=1, Direct=0`)**: questions where retrieval strictly fixes a wrong direct answer.
- **`retrieval_hurts` (`Direct=1, RAG=0`)**: questions where distracting passages break a correct direct answer.

In [ ]:
eda = df.groupby("dataset").agg(
    n_questions=("qid", "count"),
    direct_EM=("direct_em", "mean"),
    rag_EM=("rag_em", "mean"),
    direct_F1=("direct_f1", "mean"),
    rag_F1=("rag_f1", "mean"),
    retrieval_helps_rate=("retrieval_helps", "mean"),
    retrieval_helps_n=("retrieval_helps", "sum"),
    retrieval_hurts_rate=("retrieval_hurts", "mean"),
    mean_direct_logprob=("direct_logprob", "mean"),
    mean_direct_entropy=("direct_entropy", "mean"),
    mean_direct_tokens=("direct_n_tokens", "mean"),
    mean_rag_tokens=("rag_n_tokens", "mean"),
    mean_direct_lat_s=("direct_latency_s", "mean"),
    mean_rag_lat_s=("rag_latency_s", "mean"),
).round(3)
eda.to_csv(MET / "task2_eda_summary.csv")
display(eda)

# Check class balance of silver labels across splits (seed 0)
display(df.groupby(["dataset", "split_s0"])["retrieval_helps"].agg(["count", "sum", "mean"]).round(3))

In [ ]:
# Uncertainty score distributions: direct-correct vs retrieval-helps on TriviaQA vs HotpotQA
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex="col")
for row_idx, ds in enumerate(["triviaqa", "hotpotqa"]):
    sub = df[df["dataset"] == ds]
    helps = sub[sub["retrieval_helps"] == 1]
    no_help = sub[sub["retrieval_helps"] == 0]

    ax = axes[row_idx, 0]
    ax.hist(no_help["direct_logprob"].dropna(), bins=25, alpha=0.6, density=True, label="No help / direct OK")
    ax.hist(helps["direct_logprob"].dropna(), bins=25, alpha=0.6, density=True, label="Retrieval helps (RAG=1, Dir=0)")
    ax.set_title(f"{ds.upper()} · Direct mean log-probability")
    ax.set_ylabel("Density"); ax.grid(alpha=0.25); ax.legend(fontsize=8)

    ax = axes[row_idx, 1]
    ax.hist(no_help["direct_entropy"].dropna(), bins=25, alpha=0.6, density=True, label="No help / direct OK")
    ax.hist(helps["direct_entropy"].dropna(), bins=25, alpha=0.6, density=True, label="Retrieval helps (RAG=1, Dir=0)")
    ax.set_title(f"{ds.upper()} · Direct mean predictive entropy")
    ax.grid(alpha=0.25); ax.legend(fontsize=8)

axes[1, 0].set_xlabel("Length-normalised token log-probability (higher = more confident)")
axes[1, 1].set_xlabel("Mean predictive entropy in nats (higher = more uncertain)")
fig.tight_layout()
fig.savefig(FIG / "task2_score_distributions.png", dpi=150)
plt.show()

## 4. Method and fair-comparison protocol

- **Splits & seeds:** 60/20/20 train/val/test by source question, stratified by dataset, over seeds `0, 1, 2` (reported as `mean ± std` via `dsa4.evaluate.aggregate_seed_results`).
- **Threshold tuning:** For each trigger, sweep thresholds on **TriviaQA validation** (`split_s{seed} == 'val'`) and select the threshold achieving `EM >= always_RAG_EM - EM_TOL` (`EM_TOL = 0.01`) with the **lowest retrieval rate**.
- **Zero-retuning transfer:** Apply the TriviaQA-tuned threshold **unchanged** to **HotpotQA test**, and compare against an in-domain HotpotQA-tuned threshold as an upper reference.
- **Cost & token accounting (`dsa4.cost`):**
  - `CostWeights(lookup=0, retrieval=1, call_3b=3, call_7b=10)`.
  - **Post-generation triggers (V2, V3)** run the direct 7B pass first; triggered queries incur retrieval + a **second** 7B RAG pass (`tokens = direct + rag`, `cost = 10 + 1 + 10 = 21`).
  - **Pre-generation policies (V1, V4 DistilBERT)** decide *before* generation and pay a **single** 7B pass per query (`cost = 10` direct, `11` RAG), plus DistilBERT inference latency (~ms).

In [ ]:
SEEDS = (0, 1, 2)
EM_TOL = 0.01  # match always-RAG EM within 1 percentage point at minimum retrieval rate
COST_WEIGHTS = cost.CostWeights(lookup=0.0, retrieval=1.0, call_3b=3.0, call_7b=10.0)
print("Cost weights:", COST_WEIGHTS.as_dict(), "| EM tolerance:", EM_TOL)

## 5. Offline baselines: V1 (fixed), V2 (log-probability), and V3 (entropy)

In [ ]:
offline_seed_df, offline_summary = retrieval.evaluate_triggers_across_seeds(
    df, seeds=SEEDS, em_tolerance=EM_TOL, weights=COST_WEIGHTS
)
evaluate.save_results_table(offline_seed_df, MET / "task2_offline_by_seed.csv")
evaluate.save_results_table(offline_summary, MET / "task2_offline_summary.csv")

cols_show = ["eval_set", "method", "n_seeds", "threshold", "EM", "F1",
             "retrieval_rate", "auroc_helps", "retrievals_per_1k", "tokens_per_1k", "cost_per_1k"]
display(evaluate.format_results_table(offline_summary)[cols_show])

In [ ]:
# Seed-0 EM vs retrieval-rate curves for offline triggers (V2 & V3) on TriviaQA & HotpotQA test
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=False)
for ax, ds, title in [
    (axes[0], "triviaqa", "TriviaQA test (in-domain, seed 0)"),
    (axes[1], "hotpotqa", "HotpotQA test (zero-retuning transfer, seed 0)"),
]:
    tq_val = df[(df["dataset"] == "triviaqa") & (df["split_s0"] == "val")]
    test_sub = df[(df["dataset"] == ds) & (df["split_s0"] == "test")]

    for label, col, direction, color in [
        ("V2 Log-prob", "direct_logprob", "below", "#1f77b4"),
        ("V3 Entropy", "direct_entropy", "above", "#ff7f0e"),
    ]:
        c_val = retrieval.sweep_thresholds(tq_val, tq_val[col], direction=direction, is_pre_generation=False, weights=COST_WEIGHTS)
        t_star = retrieval.pick_threshold(c_val, always_rag_em=float(tq_val["rag_em"].mean()), em_tolerance=EM_TOL)
        c_test = retrieval.sweep_thresholds(test_sub, test_sub[col], direction=direction, is_pre_generation=False, weights=COST_WEIGHTS)
        op = retrieval.evaluate_policy(test_sub, retrieval.apply_threshold(test_sub[col], t_star, direction), is_pre_generation=False, weights=COST_WEIGHTS)
        ax.plot(c_test["retrieval_rate"] * 100, c_test["EM"] * 100, label=label, color=color, lw=2)
        ax.scatter([op["retrieval_rate"] * 100], [op["EM"] * 100], color=color, edgecolor="black", s=65, zorder=5)

    never_em = float(test_sub["direct_em"].mean()) * 100
    always_em = float(test_sub["rag_em"].mean()) * 100
    oracle = retrieval.evaluate_oracle(test_sub, weights=COST_WEIGHTS)
    ax.axhline(always_em, color="grey", ls="--", lw=1.2, label=f"V1b Always RAG ({always_em:.1f}%)")
    ax.axhline(never_em, color="grey", ls=":", lw=1.2, label=f"V1a Never RAG ({never_em:.1f}%)")
    ax.scatter([oracle["retrieval_rate"] * 100], [oracle["EM"] * 100], marker="*", s=140, color="green", zorder=6, label="Oracle")
    ax.set_title(title); ax.set_xlabel("Retrieval rate (%)"); ax.set_ylabel("Exact Match (%)")
    ax.grid(alpha=0.25); ax.legend(fontsize=8, loc="lower right")

fig.tight_layout()
fig.savefig(FIG / "task2_offline_curves.png", dpi=150)
plt.show()

## 6. V4: Fine-tuned DistilBERT-base (66M) complexity classifier

Simplified from **Jeong et al. (2024, Adaptive-RAG)** into a binary pre-generation router: `distilbert-base-uncased` takes only the **question text** and predicts whether retrieval helps (`rag_em == 1` and `direct_em == 0`).

- Trained with **balanced class weights** on the training split only (`split_s{seed} == 'train'`).
- **Backup rule (from Modeling Tasks):** if the TriviaQA training split has fewer than `MIN_POS = 30` positive `retrieval_helps` examples, automatically pools TriviaQA + HotpotQA training splits with class weighting (logged explicitly below); otherwise trains purely on TriviaQA train to test true zero-shot transfer to HotpotQA.

In [ ]:
DISTILBERT_MODEL = "distilbert-base-uncased"
EPOCHS = 4
BATCH_SIZE = 16
LR = 2e-5
MIN_POS = 30
POOL_IF_FEW_POSITIVES = True

v4_probs_by_seed = {}
v4_latency_ms_by_seed = {}
v4_histories = []
v4_pred_rows = []
tfidf_aurocs = []

for seed in SEEDS:
    scol = f"split_s{seed}"
    tq_train = df[(df["dataset"] == "triviaqa") & (df[scol] == "train")].reset_index(drop=True)
    tq_val = df[(df["dataset"] == "triviaqa") & (df[scol] == "val")].reset_index(drop=True)

    n_pos = int(tq_train["retrieval_helps"].sum())
    if POOL_IF_FEW_POSITIVES and n_pos < MIN_POS:
        train_df = df[df[scol] == "train"].reset_index(drop=True)
        val_df = df[df[scol] == "val"].reset_index(drop=True)
        train_regime = f"pooled (TriviaQA pos={n_pos} < {MIN_POS})"
    else:
        train_df, val_df = tq_train, tq_val
        train_regime = f"TriviaQA-only (pos={n_pos}/{len(tq_train)})"

    print(f"[Seed {seed}] Training V4 DistilBERT ({train_regime}) ...")
    model, tok, hist = retrieval.train_distilbert_trigger(
        train_df,
        val_df,
        target_col="retrieval_helps",
        model_name=DISTILBERT_MODEL,
        seed=seed,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        use_class_weight=True,
    )
    hist["seed"] = seed
    hist["train_regime"] = train_regime
    v4_histories.append(hist)

    # Score all questions (eval uses only val for tuning and test for reporting)
    probs, lat_ms = retrieval.predict_distilbert_proba(model, tok, df["question"].tolist(), batch_size=64)
    v4_probs_by_seed[seed] = dict(zip(df["qid"], probs))
    v4_latency_ms_by_seed[seed] = lat_ms

    for qid, ds, sp, y_h, p in zip(df["qid"], df["dataset"], df[scol], df["retrieval_helps"], probs):
        v4_pred_rows.append({"seed": seed, "qid": qid, "dataset": ds, "split": sp,
                             "retrieval_helps": int(y_h), "v4_prob": float(p)})

    # Quick lexical sanity check (TF-IDF + balanced LogisticRegression)
    tfidf = retrieval.train_tfidf_trigger(train_df, target_col="retrieval_helps", seed=seed)
    for ds in ("triviaqa", "hotpotqa"):
        sub = df[(df["dataset"] == ds) & (df[scol] == "test")]
        tp, _ = retrieval.predict_tfidf_proba(tfidf, sub["question"].tolist())
        vp = np.array([v4_probs_by_seed[seed][q] for q in sub["qid"]])
        tfidf_aurocs.append({
            "seed": seed, "dataset": ds,
            "tfidf_auroc": retrieval.compute_trigger_auroc(sub, tp),
            "distilbert_auroc": retrieval.compute_trigger_auroc(sub, vp),
            "v4_infer_ms": lat_ms,
        })

    # Free GPU memory cleanly between seeds
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

v4_hist_df = pd.concat(v4_histories, ignore_index=True)
v4_hist_df.to_csv(MET / "task2_v4_training_history.csv", index=False)
pd.DataFrame(v4_pred_rows).to_csv(MET / "task2_v4_predictions.csv", index=False)
display(v4_hist_df.groupby("epoch")[["train_loss", "val_loss", "val_auroc"]].mean().round(4))
display(pd.DataFrame(tfidf_aurocs).groupby("dataset")[["tfidf_auroc", "distilbert_auroc", "v4_infer_ms"]].agg(["mean", "std"]).round(3))

## 7. Full comparison & TriviaQA → HotpotQA transfer (V1–V4, seeds 0–2)

All methods evaluated across seeds `0, 1, 2` with thresholds tuned strictly on TriviaQA validation and transferred unchanged to HotpotQA test (plus HotpotQA in-domain upper reference).

In [ ]:
all_seed_df, all_summary = retrieval.evaluate_triggers_across_seeds(
    df,
    seeds=SEEDS,
    em_tolerance=EM_TOL,
    v4_probs_by_seed=v4_probs_by_seed,
    v4_latency_ms_by_seed=v4_latency_ms_by_seed,
    weights=COST_WEIGHTS,
)
evaluate.save_results_table(all_seed_df, MET / "task2_results_by_seed.csv")
evaluate.save_results_table(all_summary, MET / "task2_summary.csv")
evaluate.save_results_table(all_summary, MET / "task2_summary_clean.csv", formatted_only=True)

clean_tbl = evaluate.format_results_table(all_summary)
main_cols = ["eval_set", "method", "threshold", "EM", "F1", "retrieval_rate",
             "auroc_helps", "retrievals_per_1k", "tokens_per_1k", "cost_per_1k", "mean_latency_ms", "p95_latency_ms"]
print("=== TriviaQA Test (In-Domain) ===")
display(clean_tbl[clean_tbl["eval_set"] == "TriviaQA (in-domain)"][main_cols])

print("=== HotpotQA Test: Zero-Retuning Transfer vs In-Domain Reference ===")
transfer_tbl = clean_tbl[clean_tbl["eval_set"].str.startswith("HotpotQA")][main_cols]
transfer_tbl.to_csv(MET / "task2_transfer_table.csv", index=False)
display(transfer_tbl)

In [ ]:
# Main slide figure: EM & F1 vs Retrieval Rate for V1-V4 (TriviaQA test & HotpotQA transfer)
fig, axes = plt.subplots(2, 2, figsize=(12, 8.2), sharex=True)
palette = {
    "V2 Log-probability": "#1f77b4",
    "V3 Predictive entropy": "#ff7f0e",
    "V4 DistilBERT classifier": "#9467bd",
}

seed = 0
tq_val = df[(df["dataset"] == "triviaqa") & (df[f"split_s{seed}"] == "val")].reset_index(drop=True)

for col_idx, (ds, ds_title) in enumerate([
    ("triviaqa", "TriviaQA test (in-domain)"),
    ("hotpotqa", "HotpotQA test (TriviaQA threshold, zero retuning)"),
]):
    test_sub = df[(df["dataset"] == ds) & (df[f"split_s{seed}"] == "test")].reset_index(drop=True)
    specs = [
        ("V2 Log-probability", tq_val["direct_logprob"], test_sub["direct_logprob"], "below", False, 0.0),
        ("V3 Predictive entropy", tq_val["direct_entropy"], test_sub["direct_entropy"], "above", False, 0.0),
        ("V4 DistilBERT classifier",
         np.array([v4_probs_by_seed[seed][q] for q in tq_val["qid"]]),
         np.array([v4_probs_by_seed[seed][q] for q in test_sub["qid"]]),
         "above", True, v4_latency_ms_by_seed[seed]),
    ]

    for row_idx, metric in enumerate(["EM", "F1"]):
        ax = axes[row_idx, col_idx]
        for name, val_s, test_s, direction, pre_gen, clf_ms in specs:
            c_val = retrieval.sweep_thresholds(tq_val, val_s, direction=direction, is_pre_generation=pre_gen, classifier_latency_ms=clf_ms, weights=COST_WEIGHTS)
            t_star = retrieval.pick_threshold(c_val, always_rag_em=float(tq_val["rag_em"].mean()), em_tolerance=EM_TOL)
            c_test = retrieval.sweep_thresholds(test_sub, test_s, direction=direction, is_pre_generation=pre_gen, classifier_latency_ms=clf_ms, weights=COST_WEIGHTS)
            op = retrieval.evaluate_policy(test_sub, retrieval.apply_threshold(test_s, t_star, direction), is_pre_generation=pre_gen, classifier_latency_ms=clf_ms, weights=COST_WEIGHTS)
            ax.plot(c_test["retrieval_rate"] * 100, c_test[metric] * 100, label=name, color=palette[name], lw=2)
            ax.scatter([op["retrieval_rate"] * 100], [op[metric] * 100], color=palette[name], edgecolor="black", s=70, zorder=5)

        never_v = float(test_sub[f"direct_{metric.lower()}"].mean()) * 100
        always_v = float(test_sub[f"rag_{metric.lower()}"].mean()) * 100
        oracle = retrieval.evaluate_oracle(test_sub, weights=COST_WEIGHTS)
        ax.axhline(always_v, color="grey", ls="--", lw=1.2, label=f"V1b Always RAG ({always_v:.1f}%)")
        ax.axhline(never_v, color="grey", ls=":", lw=1.2, label=f"V1a Never RAG ({never_v:.1f}%)")
        ax.scatter([oracle["retrieval_rate"] * 100], [oracle[metric] * 100], marker="*", s=150, color="green", zorder=6, label="Oracle")
        if row_idx == 0:
            ax.set_title(ds_title)
        if col_idx == 0:
            ax.set_ylabel(f"{metric} (%)")
        if row_idx == 1:
            ax.set_xlabel("Retrieval rate (%)")
        ax.grid(alpha=0.25)
        if row_idx == 0:
            ax.legend(fontsize=8, loc="lower right")

fig.suptitle("Task 2 · Accuracy (EM & F1) vs. Retrieval Rate (markers = TriviaQA-tuned operating points)", y=1.01)
fig.tight_layout()
fig.savefig(FIG / "task2_em_f1_vs_retrieval.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Efficiency comparison: retrievals/1k, generated tokens/1k, weighted cost/1k, latency (mean +/- std)
order = ["V1a Never retrieve", "V2 Log-probability", "V3 Predictive entropy",
         "V4 DistilBERT classifier", "V1b Always retrieve", "Oracle (cheapest correct)"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, metric, ylabel in [
    (axes[0], "retrievals_per_1k", "Retrievals / 1,000 queries"),
    (axes[1], "tokens_per_1k", "Generated tokens / 1,000 queries"),
    (axes[2], "cost_per_1k", "Weighted cost / 1,000 queries"),
]:
    x = np.arange(len(order))
    w_bar = 0.36
    for shift, eval_name, color in [
        (-w_bar / 2, "TriviaQA (in-domain)", "#1f77b4"),
        (w_bar / 2, "HotpotQA (transfer)", "#d62728"),
    ]:
        sub = all_summary[all_summary["eval_set"] == eval_name].set_index("method").reindex(order)
        ax.bar(x + shift, sub[f"{metric}_mean"], w_bar, yerr=sub[f"{metric}_std"],
               capsize=3, label=eval_name, color=color, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(" ", "\n") for m in order], fontsize=8)
    ax.set_ylabel(ylabel); ax.grid(axis="y", alpha=0.25)
axes[0].legend(fontsize=8)
fig.suptitle("Task 2 · Efficiency & Cost Breakdown (mean ± std across seeds 0–2)", y=1.02)
fig.tight_layout()
fig.savefig(FIG / "task2_efficiency_cost.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Deeper analysis: calibration, reliability diagrams & confident-but-wrong audit

To diagnose **why** thresholds transfer (or fail to transfer) from TriviaQA to HotpotQA, we evaluate probability calibration on both datasets:
1. **V2 Raw LLM confidence** (`exp(direct_logprob)` vs empirical `direct_em`) — zero fitting; tests whether Qwen2.5 is systematically overconfident on multi-hop HotpotQA questions.
2. **V2 Platt-calibrated `P(retrieval_helps | -logprob)`** — logistic calibrator fitted strictly on TriviaQA validation.
3. **V3 Platt-calibrated `P(retrieval_helps | entropy)`** — logistic calibrator fitted strictly on TriviaQA validation.
4. **V4 DistilBERT `P(retrieval_helps | question)`** — predicted probability from the fine-tuned classifier.

In [ ]:
cal_rows = []
seed0_bins = {}

for seed in SEEDS:
    scol = f"split_s{seed}"
    tq_val = df[(df["dataset"] == "triviaqa") & (df[scol] == "val")].reset_index(drop=True)
    platt_lp = retrieval.fit_platt_scaler(tq_val["neg_logprob"], tq_val["retrieval_helps"])
    platt_ent = retrieval.fit_platt_scaler(tq_val["direct_entropy"], tq_val["retrieval_helps"])

    for ds in ("triviaqa", "hotpotqa"):
        sub = df[(df["dataset"] == ds) & (df[scol] == "test")].reset_index(drop=True)
        v4_p = np.array([v4_probs_by_seed[seed][q] for q in sub["qid"]])
        checks = [
            ("V2 Raw confidence (vs direct EM)", sub["direct_em"].to_numpy(), sub["direct_conf"].to_numpy()),
            ("V2 Log-prob (Platt -> retr helps)", sub["retrieval_helps"].to_numpy(), retrieval.apply_platt_scaler(platt_lp, sub["neg_logprob"])),
            ("V3 Entropy (Platt -> retr helps)", sub["retrieval_helps"].to_numpy(), retrieval.apply_platt_scaler(platt_ent, sub["direct_entropy"])),
            ("V4 DistilBERT (P -> retr helps)", sub["retrieval_helps"].to_numpy(), v4_p),
        ]
        for label, y_true, y_prob in checks:
            res_cal = retrieval.expected_calibration_error(y_true, y_prob, n_bins=10)
            cal_rows.append({
                "dataset": ds, "model": label, "seed": seed,
                "ECE": res_cal["ece"], "MCE": res_cal["mce"], "Brier": res_cal["brier"],
            })
            if seed == 0:
                seed0_bins[(ds, label)] = res_cal

cal_seed_df = pd.DataFrame(cal_rows)
cal_summary = evaluate.aggregate_seed_results(
    cal_seed_df, group_cols=["dataset", "model"], metric_cols=["ECE", "MCE", "Brier"], decimals=3
)
evaluate.save_results_table(cal_summary, MET / "task2_calibration.csv")
display(evaluate.format_results_table(cal_summary))

In [ ]:
# Reliability diagrams (seed 0): TriviaQA test vs HotpotQA test
model_labels = [
    "V2 Raw confidence (vs direct EM)",
    "V2 Log-prob (Platt -> retr helps)",
    "V3 Entropy (Platt -> retr helps)",
    "V4 DistilBERT (P -> retr helps)",
]
fig, axes = plt.subplots(2, 4, figsize=(15, 6.8), sharex=True, sharey=True)
for r, ds in enumerate(["triviaqa", "hotpotqa"]):
    for c, label in enumerate(model_labels):
        ax = axes[r, c]
        info = seed0_bins[(ds, label)]
        bdf = info["bins_df"]
        non_empty = bdf[bdf["count"] > 0]
        ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect")
        ax.bar(bdf["bin_center"], bdf["empirical_accuracy"].fillna(0.0), width=0.09,
               alpha=0.55, edgecolor="black", label="Empirical")
        ax.plot(non_empty["avg_confidence"], non_empty["empirical_accuracy"], "o-", color="#d62728", lw=1.5)
        ax.set_title(f"{ds.upper()} · ECE={info['ece']:.3f}\n{label}", fontsize=8.5)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.25)
        if c == 0:
            ax.set_ylabel("Empirical frequency")
        if r == 1:
            ax.set_xlabel("Predicted probability")

fig.suptitle("Task 2 · Reliability Diagrams: TriviaQA vs. HotpotQA (seed 0)", y=1.01)
fig.tight_layout()
fig.savefig(FIG / "task2_reliability_diagrams.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Confident-but-wrong failure audit (questions where direct was confident & wrong, and RAG fixed it)
pd.set_option("display.max_colwidth", 90)
cw_tq = retrieval.confident_wrong_examples(df, top_k=10, dataset="triviaqa")
cw_hq = retrieval.confident_wrong_examples(df, top_k=10, dataset="hotpotqa")
cw_all = pd.concat([cw_tq, cw_hq], ignore_index=True)
cw_all.to_csv(MET / "task2_confident_wrong_examples.csv", index=False)
print("Top 5 confident-but-wrong HotpotQA examples rescued by RAG:")
display(cw_hq[["qid", "question", "direct_answer", "rag_answer", "direct_conf", "direct_entropy"]].head(5))
print("Top 5 confident-but-wrong TriviaQA examples rescued by RAG:")
display(cw_tq[["qid", "question", "direct_answer", "rag_answer", "direct_conf", "direct_entropy"]].head(5))

## 9. Shared final pipeline comparison (Tasks 1–4 vs Always-RAG vs Oracle)

Per the 6-day plan (Day 4, Manimegalai), this cell combines the winning decision policy from each task against **Always-RAG** and the **Oracle** using Vijaya's shared `dsa4.cost.CostWeights` model for the **Shared Opening slide**.

- The Task 2 row is populated automatically from this notebook's best trigger.
- On Day 4, plug in the team's final winning numbers from Tasks 1, 3, and 4 into `TEAM_WINNERS` below and re-run to export `results/metrics/final_pipeline_comparison.csv`.

In [ ]:
# Automatically extract the best Task 2 trigger (lowest cost_per_1k among triggers on TriviaQA test)
t2_candidates = all_summary[
    (all_summary["eval_set"] == "TriviaQA (in-domain)")
    & (all_summary["method"].isin(["V2 Log-probability", "V3 Predictive entropy", "V4 DistilBERT classifier"]))
].sort_values(["EM_mean", "cost_per_1k_mean"], ascending=[False, True])
best_t2_method = t2_candidates.iloc[0]["method"]
print("Best Task 2 trigger on TriviaQA:", best_t2_method)

# Pull Task 2 numbers across both datasets
def _row_for(eval_set, method):
    r = all_summary[(all_summary["eval_set"] == eval_set) & (all_summary["method"] == method)].iloc[0]
    return {"EM": r["EM_mean"], "cost_per_1k": r["cost_per_1k_mean"], "retrieval_rate": r["retrieval_rate_mean"]}

always_tq = _row_for("TriviaQA (in-domain)", "V1b Always retrieve")
oracle_tq = _row_for("TriviaQA (in-domain)", "Oracle (cheapest correct)")
best_t2_tq = _row_for("TriviaQA (in-domain)", best_t2_method)

# Plug in winning rates from Tasks 1, 3, and 4 on Day 4 (set to None if not yet merged)
TEAM_WINNERS = {
    "task1_cache_hit_rate": None,       # e.g. 0.35 (from Naveena Task 1 best rule at 20% repeats)
    "task1_unsafe_reuse": None,         # e.g. 0.018
    "task3_borderline_rate": None,      # e.g. 0.08 (from Meet Task 3)
    "task3_patch_rate": None,           # e.g. 0.03
    "task4_admission_precision": None,  # e.g. 0.90 (from Vijaya Task 4)
}

comp_rows = [
    {"system": "Always-Retrieve (no cache)", "EM": always_tq["EM"],
     "cost_per_1k": always_tq["cost_per_1k"], "unsafe_reuse": 0.0},
    {"system": f"Task 2 Best ({best_t2_method}, no cache)", "EM": best_t2_tq["EM"],
     "cost_per_1k": best_t2_tq["cost_per_1k"], "unsafe_reuse": 0.0},
    {"system": "Oracle (cheapest correct, no cache)", "EM": oracle_tq["EM"],
     "cost_per_1k": oracle_tq["cost_per_1k"], "unsafe_reuse": 0.0},
]

# When Tasks 1, 3, 4 numbers are plugged in, compute the full combined pipeline row
if TEAM_WINNERS["task1_cache_hit_rate"] is not None:
    h = TEAM_WINNERS["task1_cache_hit_rate"]
    u = TEAM_WINNERS["task1_unsafe_reuse"] or 0.0
    b = TEAM_WINNERS["task3_borderline_rate"] or 0.0
    miss_rate = max(0.0, 1.0 - h - b)
    c_lookup = cost.call_cost("lookup", COST_WEIGHTS)
    c_3b = cost.call_cost("call_3b", COST_WEIGHTS)
    c_ret = cost.call_cost("retrieval", COST_WEIGHTS)
    combined_cost = 1000.0 * (c_lookup + b * (c_ret + c_3b)) + miss_rate * best_t2_tq["cost_per_1k"]
    combined_em = h * (1.0 - u) + (1.0 - h) * best_t2_tq["EM"]
    comp_rows.insert(2, {
        "system": "Combined Pipeline (Best Tasks 1–4)",
        "EM": combined_em,
        "cost_per_1k": combined_cost,
        "unsafe_reuse": u,
    })

final_comp = pd.DataFrame(comp_rows).round(3)
final_comp.to_csv(MET / "final_pipeline_comparison.csv", index=False)
display(final_comp)

## 10. Discussion, limitations and MVP gateway integration

*Fill in your final prose after running the full notebook on Kaggle with the 2,000-question cache:*

- **Best Task 2 policy & key numbers:** State the winning trigger (`V2`, `V3`, or `V4`), its EM/F1, retrievals saved vs. always-RAG, generated tokens/1k, and AUROC (mean ± std over seeds 0–2).
- **TriviaQA → HotpotQA transfer verdict:** Summarize whether the TriviaQA-tuned threshold held on HotpotQA or under-retrieved, backed by the ECE / reliability diagram findings and confident-but-wrong examples.
- **Pre-generation vs. post-generation trade-off:** Highlight that V2/V3 incur a second 7B LLM pass on every triggered query, whereas V4 DistilBERT routes before generation with a single LLM call.
- **Limitations (dropped 'could' items):** Direct hidden-state comparison with **CtrlA (Liu et al., 2025)** was deferred as a 'could' item because hidden-state probing requires online white-box access rather than offline shared-cache evaluation; V4 uses a simplified 2-way binary router rather than 3-way multi-step Adaptive-RAG.
- **MVP gateway integration:** On a cache miss from Task 1, the gateway invokes the winning Task 2 trigger to decide between direct generation and top-5 FAISS RAG, logging retrieval rate and latency to the dashboard.

### Key references
- Moskvoretskii et al. (ACL 2025). *Adaptive Retrieval Without Self-Knowledge? Bringing Uncertainty Back Home.*
- Jeong et al. (NAACL 2024). *Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity.*
- Shah (2026). *Grounded Cache Routing for Retrieval-Augmented Generation.*
- Schroeder et al. (ICLR 2026). *vCache: Verified Semantic Prompt Caching.*